# IMPORT

In [35]:
import pandas as pd
import numpy as np
import pathlib as p
from pathlib import Path

# DEFINE FOLDERS

In [36]:
DATASET_INPUT_FOLDER = Path("correlated_data_csv")
DATASET_OUTPUT_FOLDER = Path("clean_data_csv")
DELETED_STATIONS_FOLDER = Path("deleted_stations")

folders = [DATASET_INPUT_FOLDER, DATASET_OUTPUT_FOLDER, DELETED_STATIONS_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# LOAD DATA

In [37]:
# Load data
df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/correlated_data_2.csv")


# KEEPING THE USABLE COLUMNS

In [38]:
# Target
df["total_bikes_now"] = df["numbikesavailable"]

# Keep only useful columns
keep_cols = [
    "stationcode",
    "latitude",
    "longitude",
    "capacity",
    "total_bikes_now",
    "weather_time",
    "weather_temperature_2m",
    "weather_wind_speed_10m",
    "weather_wind_direction_10m",
    "weather_weather_code",
    "weather_is_day",
]

df = df[keep_cols]

print("Initial shape:", df.shape)

Initial shape: (4907829, 11)


# WEATHER TIME HANDLING

In [39]:
df["weather_time"] = pd.to_datetime(df["weather_time"], errors="coerce")

# Drop rows without time
df = df.dropna(subset=["weather_time"])

# Time features
df["hour"] = df["weather_time"].dt.hour
df["dayofweek"] = df["weather_time"].dt.dayofweek
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

# Cyclical encoding
df["sin_hour"] = np.sin(2 * np.pi * df["hour"] / 24)
df["cos_hour"] = np.cos(2 * np.pi * df["hour"] / 24)


# DROP INVALID ROWS(DOESNT CONTAIN THE DATA FOR TOTAL_BIKES_NOW)

In [40]:
df = df.dropna(subset=["total_bikes_now"])

# FILL MISSING WEATHER VALUES

In [41]:
weather_cols = [
    "weather_temperature_2m",
    "weather_wind_speed_10m",
    "weather_wind_direction_10m",
    "weather_weather_code",
    "weather_is_day",
]

for col in weather_cols:
    df[col] = df.groupby("stationcode")[col].transform(
        lambda x: x.fillna(x.mean())
    )


# DROP BROKEN WEATHER ROWS

In [42]:
df = df.dropna(subset=weather_cols)

# DETECT/REMOVE BROKEN STATIONS

In [43]:
MIN_ROWS = 200
MIN_VARIANCE = 1e-2

station_stats = (
    df.groupby("stationcode")["total_bikes_now"]
    .agg(["count", "var"])
)

broken_stations = station_stats[
    (station_stats["count"] < MIN_ROWS) |
    (station_stats["var"] < MIN_VARIANCE)
]

valid_stations = station_stats.index.difference(broken_stations.index)


# SAVE THE BROKEN STATIONS IN A FILE

In [44]:
broken_stations.to_csv(f"{DELETED_STATIONS_FOLDER}/broken_stations.csv")
print(f"❌ Broken stations saved: {len(broken_stations)}")


❌ Broken stations saved: 28


# REMOVE THEM FROM THE DATABASE

In [45]:
df = df[df["stationcode"].isin(valid_stations)]

print("After station filtering:", df.shape)
print("Remaining stations:", df['stationcode'].nunique())

After station filtering: (4763007, 16)
Remaining stations: 1474


# CHECK

In [46]:
print(df.isna().sum().sort_values(ascending=False))

stationcode                   0
latitude                      0
longitude                     0
capacity                      0
total_bikes_now               0
weather_time                  0
weather_temperature_2m        0
weather_wind_speed_10m        0
weather_wind_direction_10m    0
weather_weather_code          0
weather_is_day                0
hour                          0
dayofweek                     0
is_weekend                    0
sin_hour                      0
cos_hour                      0
dtype: int64


# SAVE THE FINAL CLEAN DATASET

In [47]:
df.to_csv(f"{DATASET_OUTPUT_FOLDER}/clean_bike_weather_data.csv", index=False)
print("✅ Clean dataset saved: clean_bike_weather_data.csv")

✅ Clean dataset saved: clean_bike_weather_data.csv
